## What are TPUs?
The Tensor Processing Unit (TPU) is a custom integrated chip, designed specifically to accelerate the process of training machine learning models. 

## TPUs for free at Kaggle
**You can use up to 30 hours per week of TPUs and up to 9h at a time in a single session.**
**For more info you can visit [here](https://www.kaggle.com/docs/tpu).**

## Why do we need TFRecord format?
The TFRecord format is tensorflow's custom data format which is simple for storing a sequence of binary records. The advantages of using TFRecords are amazingly more efficient storage, fast I/O, self-contained files, etc. The main advantage of TPUs are faster I/O which results in faster model training.

For understanding the basics of TFRecords, please visit Ryan Holbrook notebook: [TFRecords Basics](https://www.kaggle.com/ryanholbrook/tfrecords-basics).

## Useful resources which helped me:
* https://www.tensorflow.org/tutorials/load_data/tfrecord
* https://www.kaggle.com/mgornergoogle/five-flowers-with-keras-and-xception-on-tpu
* https://towardsdatascience.com/a-practical-guide-to-tfrecords-584536bc786c
* https://cloud.google.com/blog/products/ai-machine-learning/what-makes-tpus-fine-tuned-for-deep-learning
* https://pub.towardsai.net/writing-tfrecord-files-the-right-way-7c3cee3d7b12

**In this notebook you will learn how to convert private dataset into TFRecord format.**

## Imports

In [1]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

2021-10-19 17:50:22.301789: W tensorflow/stream_executor/platform/default/dso_loader.cc:60] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /opt/conda/lib
2021-10-19 17:50:22.301897: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


## Load the data

We'll be using MNIST dataset which consists of handwritten digits, 70,000 examples. It is a subset of a larger set available from NIST. The digits have been size-normalized and centered in a fixed-size image. 

In this notebook, we'll be converting the [Digit Recognizer](https://www.kaggle.com/c/digit-recognizer) competition data, and will use it for model training using TPUs in the next notebook.

In [2]:
mnist_train = pd.read_csv('../input/digit-recognizer/train.csv')
mnist_test = pd.read_csv('../input/digit-recognizer/test.csv')

Extracting the labels and features from the train dataset

In [3]:
labels = mnist_train["label"]
features = mnist_train.drop(labels = ["label"],axis = 1)

## Normalize the dataset

The inputs with the large integer values can slow down the learning process,so it's a good practice to normalize the pixel values with values between 0 and 1.


In [4]:
features = features/255.0
mnist_test = mnist_test/255.0

## Resize the dataset and one-hot encode labels

By using one-hot encoding, we are converting each categorical values into new categorical column with binary values of 1 or 0.

In [5]:
features = features.values.reshape(-1,28,28,1)
mnist_test = mnist_test.values.reshape(-1,28,28,1)
labels = to_categorical(labels, num_classes=10)

In [6]:
mnist_test.shape

(28000, 28, 28, 1)

In [7]:
labels.shape

(42000, 10)

## Feature Creation functions

The following functions can be used to convert a value to a type compatible which takes a scalar input values and returns a tf.train.Feature.

In [8]:
def _bytes_feature(value):
    if isinstance(value, type(tf.constant(0))): # if value ist tensor
        value = value.numpy() # get value of tensor
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _float_feature(value):
    return tf.train.Feature(float_list=tf.train.FloatList(value=[value]))

def _int64_feature(value):
    return tf.train.Feature(int64_list=tf.train.Int64List(value=[value]))

def serialize_array(array):
    array = tf.io.serialize_tensor(array)
    return array

## Serializing and Writing 

Now, we'll create a dictionary to store the actual image, height, width and depth of the image and the label where we first serialize the array and then convert it to a bytes_feature.  All these `key:value` mappings make up the features for one Example.

In [9]:
def parse_single_image(image, label=None):
    if label is None:
            data = {
                'height' : _int64_feature(image.shape[0]),
                'width' : _int64_feature(image.shape[1]),
                'depth' : _int64_feature(image.shape[2]),
                'raw_image' : _bytes_feature(serialize_array(image))
    }
    else:
        data = {
                'height' : _int64_feature(image.shape[0]),
                'width' : _int64_feature(image.shape[1]),
                'depth' : _int64_feature(image.shape[2]),
                'raw_image' : _bytes_feature(serialize_array(image)),
                'label' : _bytes_feature(serialize_array(label))
        }
    out = tf.train.Example(features=tf.train.Features(feature=data))

    return out

## Writing to the TFRecord files

This function write our dataset to the TFRecord files using the `TFRecordWriter`

In [10]:
def convert_to_tfr(images, labels=None, filename:str='images'):
    filename = filename+'.tfrecords'
    writer = tf.io.TFRecordWriter(filename)
    count = 0 
    
    for index in range(len(images)):
        if labels is None:
            current_image = images[index]
            out = parse_single_image(image = current_image)
        else:
            current_image = images[index]
            current_label = labels[index]
            out = parse_single_image(image = current_image, label = current_label)
        writer.write(out.SerializeToString())
        count +=1
    writer.close()
    print(f"Wrote {count} elements to TFRecord")

In [11]:
convert_to_tfr(features, labels, filename="train_data")

2021-10-19 17:50:33.900928: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2021-10-19 17:50:33.904098: W tensorflow/stream_executor/platform/default/dso_loader.cc:60] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /opt/conda/lib
2021-10-19 17:50:33.904140: W tensorflow/stream_executor/cuda/cuda_driver.cc:326] failed call to cuInit: UNKNOWN ERROR (303)
2021-10-19 17:50:33.904168: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (aeeee1c32922): /proc/driver/nvidia/version does not exist
2021-10-19 17:50:33.906476: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operation

Wrote 42000 elements to TFRecord


In [12]:
convert_to_tfr(mnist_test, filename="test_data")

Wrote 28000 elements to TFRecord
